In [1]:
import numpy as np
import pandas as pd

from sklearn.datasets import (load_diabetes,load_breast_cancer,load_digits,load_iris)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (mean_squared_error,r2_score,accuracy_score,confusion_matrix,precision_recall_fscore_support)

In [2]:
# 1. Diabetes dataset
diabetes = load_diabetes(scaled=False)
X_diabetes = diabetes.data
y_diabetes = diabetes.target

# 2. Breast Cancer dataset
breast_cancer = load_breast_cancer()
X_breast = breast_cancer.data
y_breast = 1 - breast_cancer.target

# 3. Digits dataset
digits = load_digits()
X_digits = digits.data / 16.0
y_digits = digits.target

# 4. Iris dataset
iris = load_iris()
X_iris = iris.data
y_iris = iris.target


In [4]:
def split_dataset(X, y, classification=True):

    #20% for testing
    stratify_value = y if classification else None
    
    X_temp, X_test, y_temp, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=stratify_value)


    #25% of the remaining 80% for validation
    stratify_value = y_temp if classification else None

    X_train, X_val, y_train, y_val = train_test_split(
        X_temp,
        y_temp,
        test_size=0.25,
        random_state=42,
        stratify=stratify_value)

    return X_train, X_val, X_test, y_train, y_val, y_test


diabetes_split = split_dataset(X_diabetes, y_diabetes, classification=False)
breast_split = split_dataset(X_breast, y_breast)
digits_split = split_dataset(X_digits, y_digits)
iris_split = split_dataset(X_iris, y_iris)


In [5]:
datasets = {
    "Diabetes": diabetes_split,
    "Breast Cancer": breast_split,
    "Digits": digits_split,
    "Iris": iris_split
}

for name, split in datasets.items():

    X_train, X_val, X_test, y_train, y_val, y_test = split

    print("\n", name)

    print("Training size:", len(X_train))
    print("Validation size:", len(X_val))
    print("Testing size:", len(X_test))
    print("Total size:", len(X_train) + len(X_val) + len(X_test))

    if name != "Diabetes":
        print("\nTraining class counts:",
              np.unique(y_train, return_counts=True))

        print("Validation class counts:",
              np.unique(y_val, return_counts=True))

        print("Testing class counts:",
              np.unique(y_test, return_counts=True))




 Diabetes
Training size: 264
Validation size: 89
Testing size: 89
Total size: 442

 Breast Cancer
Training size: 341
Validation size: 114
Testing size: 114
Total size: 569

Training class counts: (array([0, 1]), array([214, 127]))
Validation class counts: (array([0, 1]), array([71, 43]))
Testing class counts: (array([0, 1]), array([72, 42]))

 Digits
Training size: 1077
Validation size: 360
Testing size: 360
Total size: 1797

Training class counts: (array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9]), array([106, 109, 107, 109, 109, 109, 109, 107, 104, 108]))
Validation class counts: (array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9]), array([36, 37, 35, 37, 36, 36, 36, 36, 35, 36]))
Testing class counts: (array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9]), array([36, 36, 35, 37, 36, 37, 36, 36, 35, 36]))

 Iris
Training size: 90
Validation size: 30
Testing size: 30
Total size: 150

Training class counts: (array([0, 1, 2]), array([30, 30, 30]))
Validation class counts: (array([0, 1, 2]), array([10, 10, 10]))
Testing class c

In [ ]:
class Standardizer:

    def fit(self, X):
        self.mean = np.mean(X, axis=0)
        self.std = np.std(X, axis=0)
        self.std[self.std == 0] = 1  #avoid devision by 0
        return self

    def transform(self, X):
        return (X - self.mean) / self.std

    

In [8]:
def add_bias(X):
        # Add a column of ones at the beginning
        return np.column_stack((np.ones(X.shape[0]), X))

In [9]:
X_train, X_val, X_test, y_train, y_val, y_test = diabetes_split

scaler = Standardizer()
scaler.fit(X_train)

X_train_std = scaler.transform(X_train)
X_val_std = scaler.transform(X_val)
X_test_std = scaler.transform(X_test)

print("Training means:")
print(np.mean(X_train_std, axis=0))

print("\nTraining standard deviations:")
print(np.std(X_train_std, axis=0))

X_train_bias = add_bias(X_train_std)

print("\nShape before adding bias:", X_train_std.shape)
print("Shape after adding bias:", X_train_bias.shape)


Training means:
[-2.26249995e-16  1.79990702e-16 -2.23726761e-15 -3.61663561e-16
 -5.88754634e-17 -1.08348348e-17  1.94394164e-16 -4.03717464e-17
  2.03877319e-15 -5.53100810e-16]

Training standard deviations:
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]

Shape before adding bias: (264, 10)
Shape after adding bias: (264, 11)


In [10]:
def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def rmse(y_true, y_pred):
    return np.sqrt(mse(y_true, y_pred))

def r2(y_true, y_pred):

    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_total = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - (ss_res / ss_total)

In [11]:
def confusion_matrix(y, pred, K):
    cm = np.zeros((K, K), dtype=int)
    np.add.at(cm, (y, pred), 1)
    return cm

In [12]:
def accuracy(y, pred):
    return np.mean(y == pred)


def classification_metrics(cm):

    tp = np.diag(cm)
    fp = np.sum(cm, axis=0) - tp
    fn = np.sum(cm, axis=1) - tp

    precision = np.divide(
        tp,
        tp + fp,
        out=np.zeros_like(tp, dtype=float),
        where=(tp + fp) != 0
    )

    recall = np.divide(
        tp,
        tp + fn,
        out=np.zeros_like(tp, dtype=float),
        where=(tp + fn) != 0
    )

    f1 = np.divide(
        2 * precision * recall,
        precision + recall,
        out=np.zeros_like(precision, dtype=float),
        where=(precision + recall) != 0
    )

    macro_precision = np.mean(precision)
    macro_recall = np.mean(recall)
    macro_f1 = np.mean(f1)

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1
    }